# Insect detection pipeline: one site per run

Faster R-CNN (ResNet50-FPN v2, COCO-pretrained), in two versions:
- **tiled:** 1280 px overlapping tiles from the half-resolution frame (5120 × 2880).
- **downscaled:** the whole frame at 2560 × 1440.

**How to run:** set `SITE` in the next cell, then Run All. Run the notebook once per site (Station_1st_night, then Station_2nd_night). Each site writes its results to its own work folder.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))          # this notebook sits next to the scripts
from config import CFG, paths
cfg = dict(CFG)

# ---- CHOOSE THE SITE, AND CHECK THE BASE FOLDER ----
SITE = "Station_1st_night"                        # or "Station_2nd_night" (later: "OAK")
BASE = r"D:\datasett"                             # folder that holds cvat_upload\ and annotations\

cfg["raw_dir"]   = os.path.join(BASE, "cvat_upload", SITE)                 # the 120 half-res frames (made in the next cell)
cfg["cvat_coco"] = os.path.join(BASE, "annotations", SITE + "_coco.json")  # CVAT export (COCO 1.0), renamed
cfg["work_dir"]  = os.path.join(BASE, "work", SITE)                        # outputs go here

# On Windows Jupyter, data-loader workers sometimes hang: use 0 there.
cfg["num_workers"] = 0 if os.name == "nt" else 2
assert os.path.exists(cfg["cvat_coco"]), "not found: %s" % cfg["cvat_coco"]
P = paths(cfg)
print(SITE, "| outputs ->", P["work"])


## Make the half-resolution frames (first run only)
Copies the 120 annotated frames of this site from the full-size originals at half resolution, exactly as they were made for CVAT. Set `ORIG_DIR` to where the full-size frames of this site are on this PC.

In [ ]:
ORIG_DIR = os.path.join(BASE, SITE)               # full-size originals of this SITE (change if they are elsewhere)
import make_half_res
make_half_res.main(ORIG_DIR, cfg["cvat_coco"], cfg["raw_dir"])

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - training on CPU will not finish today")
if torch.cuda.is_available():
    print("GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 0 · Check the frames
Lists every frame in the site folder and marks which ones are intact.

In [ ]:
import s00_check_frames
_ = s00_check_frames.main(cfg)

## 2 · Validate the CVAT export and measure the insects

In [ ]:
import s02_validate
stats = s02_validate.main(cfg)
# Use the anchor sizes measured from your boxes (comment this line out to keep the defaults)
cfg["anchor_sizes"] = stats["anchor_suggestion"]

## 3 · Train / val / test split in time blocks

In [ ]:
import s03_split
split = s03_split.main(cfg)

## 4 · Tiles (tiled model) and downscaled frames (baseline)

In [ ]:
import s04_tiles
s04_tiles.main(cfg)

## 5 · Train both models
The first epoch prints an estimate of the total time. Each model stops at its time cap (`max_minutes`), and the best epoch on val is kept.

In [ ]:
import s05_train
from IPython.display import Image, Markdown, display
s05_train.main(cfg, mode="tiled")
display(Image(str(P["models"] / "tiled_history.png")))

In [ ]:
s05_train.main(cfg, mode="down")
display(Image(str(P["models"] / "down_history.png")))

## 6 · Detect on whole val and test frames
Val is used only to choose the counting threshold. Test is scored once.

In [ ]:
import s06_infer
for mode in ("tiled", "down"):
    for part in ("val", "test"):
        s06_infer.main(cfg, mode=mode, frames=part)

In [ ]:
# Optional (~15 min): count insects on every good frame of the night -> activity curve for the slides
s06_infer.main(cfg, mode="tiled", frames="all")

## 7 · Results

In [ ]:
import s07_evaluate
res = s07_evaluate.main(cfg)
display(Markdown((P["results"] / "summary.md").read_text()))
for f in ("pr_curve.png", "count_scatter.png", "count_over_night_tiled.png"):
    if (P["results"] / f).exists():
        display(Image(str(P["results"] / f)))

## 8 · What went wrong: errors sorted by cause, plus example frames

In [ ]:
import s08_errors, glob
for mode in ("tiled", "down"):
    s08_errors.main(cfg, mode=mode)
for f in sorted(glob.glob(str(P["results"] / "errors_tiled" / "*.jpg"))):
    print(os.path.basename(f)); display(Image(f, width=900))

In [ ]:
for f in sorted(glob.glob(str(P["results"] / "examples_tiled" / "*.jpg")))[:3]:
    print(os.path.basename(f)); display(Image(f, width=1100))